# Day 38 — PyTorch LSTM Refresher

Embeddings → LSTM → classification head on a **toy dataset** before applying
the same architecture to real CAPEv2 API sequences on Days 40-41.

**Goal:** confirm the full pipeline works end-to-end:
1. Vocabulary + tokenization
2. Embedding layer
3. LSTM encoder
4. Linear classification head
5. Training loop with cross-entropy loss
6. Evaluation with accuracy

**Toy task:** classify short synthetic "API sequences" as benign (0) or
malicious (1). The sequences are hand-crafted to have obvious patterns,
so the model should reach ~100% accuracy quickly.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random

random.seed(42)
torch.manual_seed(42)
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Synthetic Toy Dataset

Benign sequences contain mostly file reads, registry reads, UI calls.
Malicious sequences contain process injection, network, and persistence calls.

In [ ]:
BENIGN_APIS = ['LdrLoadDll', 'RegOpenKeyExW', 'RegQueryValueExW',
               'NtCreateFile', 'NtReadFile', 'CreateWindowExW', 'ShowWindow']

MALICIOUS_APIS = ['NtAllocateVirtualMemory', 'NtWriteVirtualMemory',
                  'CreateRemoteThread', 'InternetOpenUrlA',
                  'HttpSendRequestA', 'RegSetValueExW',
                  'CreateProcessInternalW']

SHARED_APIS = ['LdrLoadDll', 'NtCreateFile']

ALL_APIS = sorted(set(BENIGN_APIS + MALICIOUS_APIS + SHARED_APIS))

PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for api in ALL_APIS:
    vocab[api] = len(vocab)

print(f'Vocabulary size: {len(vocab)}')
print(f'Tokens: {list(vocab.keys())}')

In [ ]:
def generate_sample(label, min_len=8, max_len=20):
    length = random.randint(min_len, max_len)
    if label == 0:  # benign
        seq = [random.choice(BENIGN_APIS) for _ in range(length)]
    else:  # malicious
        seq = [random.choice(SHARED_APIS)] * 2  # start normal
        seq += [random.choice(MALICIOUS_APIS) for _ in range(length - 2)]
    return seq

def tokenize(seq, vocab, max_len=25):
    ids = [vocab.get(api, vocab[UNK_TOKEN]) for api in seq[:max_len]]
    ids += [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

N_SAMPLES = 200
MAX_LEN = 25

data = []
for _ in range(N_SAMPLES // 2):
    seq = generate_sample(0)
    data.append((tokenize(seq, vocab, MAX_LEN), 0))
for _ in range(N_SAMPLES // 2):
    seq = generate_sample(1)
    data.append((tokenize(seq, vocab, MAX_LEN), 1))

random.shuffle(data)
split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]
print(f'Train: {len(train_data)}, Test: {len(test_data)}')

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, data):
        self.seqs = torch.tensor([d[0] for d in data], dtype=torch.long)
        self.labels = torch.tensor([d[1] for d in data], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.seqs[idx], self.labels[idx]

train_ds = SeqDataset(train_data)
test_ds = SeqDataset(test_data)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

## 2. LSTM Classifier Architecture

```
Input token IDs → Embedding → LSTM → last hidden state → Linear → 2 classes
```

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.0, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)           # (batch, seq_len, embed_dim)
        _, (h_n, _) = self.lstm(emb)      # h_n: (num_layers, batch, hidden_dim)
        last_hidden = h_n[-1]             # (batch, hidden_dim)
        logits = self.fc(last_hidden)     # (batch, num_classes)
        return logits

model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=16,
    hidden_dim=32,
    num_classes=2,
    pad_idx=0
)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal parameters: {total_params:,}')

## 3. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 15

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for seqs, labels in train_loader:
        optimizer.zero_grad()
        logits = model(seqs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)

    avg_loss = total_loss / total
    train_acc = correct / total

    if epoch % 3 == 0 or epoch == 1:
        print(f'Epoch {epoch:>2}/{EPOCHS}  loss={avg_loss:.4f}  train_acc={train_acc:.3f}')

## 4. Evaluation

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for seqs, labels in test_loader:
        logits = model(seqs)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)

test_acc = correct / total
print(f'Test accuracy: {test_acc:.3f} ({correct}/{total})')
print()
if test_acc >= 0.95:
    print('The model easily separates the toy classes — the pipeline works.')
else:
    print('Accuracy lower than expected on this toy task — check the data generation.')

## Refresher Notes

### Architecture choices for Day 40 (real LSTM):
- **Embedding dim**: 64 (real vocab is larger — ~200-400 unique APIs)
- **Hidden dim**: 128 (more capacity for real patterns)
- **Layers**: 2 with dropout=0.3 to regularize
- **Sequence length**: 2048 max (per feature spec), but most are <500
- **Batch size**: 64, learning rate: 1e-3 with scheduler

### Key concepts confirmed:
1. **Embedding** maps discrete token IDs to dense vectors — same idea as word2vec
2. **LSTM** processes the sequence step-by-step, maintaining a hidden state
3. We use the **last hidden state** as the sequence representation
4. A **linear head** maps hidden → class logits
5. **CrossEntropyLoss** combines log_softmax + NLLLoss
6. **padding_idx=0** ensures PAD tokens get zero gradients

### What the Transformer (Day 41) changes:
- Replace LSTM with multi-head self-attention + positional encoding
- Use mean pooling over sequence positions instead of last hidden state
- Potentially better at capturing long-range dependencies in API sequences